# LangChain: Evaluation (Modernized with LCEL)

## Outline:

* Example generation
* Manual evaluation (and debugging)
* LLM-assisted evaluation

> **Note**: This notebook uses modern LCEL chains and `langchain_core`/`langchain_openai` imports instead of deprecated `RetrievalQA`, `VectorstoreIndexCreator`, `QAGenerateChain`, and `QAEvalChain`.

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
import os
from pathlib import Path
data_dir =Path(os.getcwd()).parent /"data"

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [3]:
# Set the model variable
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

## Create our QandA application

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [5]:
file = data_dir / 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8')
data = loader.load()

In [6]:
# Create vector store and retriever (replaces VectorstoreIndexCreator + DocArrayInMemorySearch)
embeddings = OpenAIEmbeddings()
vectorstore = InMemoryVectorStore.from_documents(data, embeddings)
retriever = vectorstore.as_retriever()

In [7]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)

def format_docs(docs):
    return "<<<<>>>>>".join(doc.page_content for doc in docs)

qa_prompt = ChatPromptTemplate.from_template(
    "Use the following pieces of context to answer the question at the end. "
    "If you don't know the answer, just say that you don't know, "
    "don't try to make up an answer.\n\n"
    "{context}\n\n"
    "Question: {question}\n"
    "Helpful Answer:"
)

# LCEL RAG chain (replaces RetrievalQA)
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt
    | llm
    | StrOutputParser()
)

### Coming up with test datapoints

In [8]:
data[10]

Document(metadata={'source': 'e:\\GIT_ROOT\\Learning\\DLAI-shortcourse_notebooks\\courses\\C11 - LangChain for LLM Application Development\\data\\OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [9]:
data[11]

Document(metadata={'source': 'e:\\GIT_ROOT\\Learning\\DLAI-shortcourse_notebooks\\courses\\C11 - LangChain for LLM Application Development\\data\\OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

### Hard-coded examples

In [10]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

### LLM-Generated examples

In [11]:
import json

# LCEL chain to generate QA examples (replaces deprecated QAGenerateChain)
qa_gen_prompt = ChatPromptTemplate.from_template(
    "Given the following document, generate a question and answer pair. "
    "The question should be specific to the document content. "
    "Return a JSON object with keys 'qa_pairs' containing a single object with 'query' and 'answer' keys.\n\n"
    "Document:\n{doc}\n\n"
    "Return ONLY the JSON, no other text."
)

qa_gen_chain = qa_gen_prompt | ChatOpenAI(model=llm_model, temperature=0.0) | StrOutputParser()

In [17]:
import re

def parse_json_response(text: str) -> dict:
    """Strip markdown code fences and parse JSON."""
    cleaned = re.sub(r"```(?:json)?\s*", "", text).strip()
    return json.loads(cleaned)

# Generate QA pairs from first 5 documents (replaces QAGenerateChain.apply_and_parse)
new_examples = []
for doc in data[:5]:
    result = qa_gen_chain.invoke({"doc": doc.page_content})
    try:
        parsed = parse_json_response(result)
        qa_pair = parsed.get("qa_pairs", parsed)
        # Handle both list and dict formats
        if isinstance(qa_pair, list):
            qa_pair = qa_pair[0]
        new_examples.append({"query": qa_pair["query"], "answer": qa_pair["answer"]})
    except (json.JSONDecodeError, KeyError, IndexError, TypeError) as e:
        print(f"Skipping doc due to parse error: {e}\nRaw output: {result[:200]}")

In [18]:
new_examples[0]

{'query': "What material is used for the Women's Campside Oxfords to provide a broken-in feel?",
 'answer': "The Women's Campside Oxfords use soft canvas material for a broken-in feel and look."}

In [19]:
data[0]

Document(metadata={'source': 'e:\\GIT_ROOT\\Learning\\DLAI-shortcourse_notebooks\\courses\\C11 - LangChain for LLM Application Development\\data\\OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

### Combine examples

In [20]:
examples += new_examples

In [21]:
# invoke() replaces deprecated .run()
qa_chain.invoke(examples[0]["query"])

'Yes, the Cozy Comfort Pullover Set has side pockets.'

## Manual Evaluation

In [22]:
from langchain_core.globals import set_debug
set_debug(True)

In [23]:
qa_chain.invoke(examples[0]["query"])

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [

'Yes, the Cozy Comfort Pullover Set has side pockets.'

In [24]:
# Turn off debug mode
set_debug(False)

## LLM assisted evaluation

In [25]:
# Generate predictions for all examples (replaces qa.apply())
predictions = []
for ex in examples:
    result = qa_chain.invoke(ex["query"])
    predictions.append({
        "query": ex["query"],
        "answer": ex["answer"],
        "result": result,
    })

In [26]:
# LLM-assisted evaluation chain (replaces deprecated QAEvalChain)
eval_prompt = ChatPromptTemplate.from_template(
    "You are grading a student's answer to a question.\n\n"
    "QUESTION: {query}\n"
    "CORRECT ANSWER: {answer}\n"
    "STUDENT ANSWER: {result}\n\n"
    "Grade the student's answer as CORRECT or INCORRECT. "
    "Respond with only 'CORRECT' or 'INCORRECT'."
)

In [27]:
llm = ChatOpenAI(temperature=0, model=llm_model)
eval_chain = eval_prompt | llm | StrOutputParser()

In [28]:
# Evaluate all predictions
graded_outputs = []
for pred in predictions:
    grade = eval_chain.invoke({
        "query": pred["query"],
        "answer": pred["answer"],
        "result": pred["result"],
    })
    graded_outputs.append({"text": grade})

In [29]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set has side pockets.
Predicted Grade: CORRECT

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: CORRECT

Example 2:
Question: What material is used for the Women's Campside Oxfords to provide a broken-in feel?
Real Answer: The Women's Campside Oxfords use soft canvas material for a broken-in feel and look.
Predicted Answer: The Women's Campside Oxfords use soft canvas material to provide a broken-in feel.
Predicted Grade: CORRECT

Example 3:
Question: What are the dimensions of the medium-sized Recycled Waterhog Dog Mat?
Real Answer: The medium-sized Recycled Waterhog Dog Mat has dimensions of 22.5" x 34.5".
Predicted Answer: The dimensions of

In [30]:
graded_outputs[0]

{'text': 'CORRECT'}